#### 1. Obtain protein IDs of orthologs per GENE from NCBI
#### 2. Using the IDS Fetch protein sequence ID from 'records' from the [NCBI query page](https://www.ncbi.nlm.nih.gov/labs/gquery/) 
#### 3. Store nested dictionary of protein : sequences
#### 4. identify and extract unique sequences, human first then non-human
#### 5. Save sequences in FASTA format for each gene in /Fasta folder


#### 

In [1]:
run Kondrashov

* function: fetch protein record of each ortholog from NCBI

In [2]:
def fetch_gene_record(gene_id):
    """
    Fetch one Entrez Gene XML record and parse it with xmltodict.
    """
    handle = Entrez.efetch(db="gene", id=gene_id, rettype="xml", retmode="text")
    record = x2d.parse(handle.read().decode("utf-8"))
    handle.close()
    return record

* function: extract protein ID from dictionary into a 'list'

In [3]:
def extract_protein_accessions(obj):
    """
    Recursively search the NCBI Gene XML dictionary and collect protein accessions.
    Returns accessions like XP_077799762.1 or NP_000148.2.
    """
    proteins = set()

    if isinstance(obj, dict):
        acc = obj.get("Gene-commentary_accession")
        ver = obj.get("Gene-commentary_version")

        if acc is not None:
            # Protein accessions usually start with NP_, XP_, or YP_
            if acc.startswith(("NP_", "XP_", "YP_")):
                if ver is not None:
                    proteins.add(f"{acc}.{ver}")
                else:
                    proteins.add(acc)

        for value in obj.values():
            proteins.update(extract_protein_accessions(value))

    elif isinstance(obj, list):
        for item in obj:
            proteins.update(extract_protein_accessions(item))

    return proteins


### [Rodentia](https://meshb.nlm.nih.gov/record/ui?ui=D012377&dcmsLinks=true)  txid9989
* RUN CODE USING DEFINED FUNCTIONS: obtain orthologs per gene 
* final output is 'records' contains gene, rodentia orthologs, details

In [ ]:
# obtain rodentia Orthologs per gene
result = {}

for locus in lociii:
    query = f"{locus}[Gene Name] AND txid9989[Organism]"
    handle = Entrez.esearch(db="gene", term=query, retmax=100)
    search_record = Entrez.read(handle)
    handle.close()

    gene_ids = search_record["IdList"]
    print(f"{locus}: {search_record['Count']} -> {gene_ids[:4]}")


    result[locus] = {}

    for gene_id in gene_ids:
        try:
            gene_record = fetch_gene_record(gene_id)
            protein_ids = sorted(extract_protein_accessions(gene_record))
            result[locus][gene_id] = protein_ids
            print(f"  {gene_id}: {len(protein_ids)} proteins")

            # NCBI's request limit (10/sec with an API key)
            time.sleep(0.1)

        except Exception as expt:
            print(f"  Error with {locus} / {gene_id}: {expt}")
            result[locus][gene_id] = []

ABCA12: 52 -> ['74591', '301482', '311596418', '308630673']
  74591: 1 proteins
  301482: 3 proteins
  311596418: 1 proteins
  308630673: 2 proteins
  308432974: 2 proteins
  308258540: 2 proteins
  306656924: 1 proteins
  143407014: 2 proteins
  142837006: 1 proteins
  138830722: 2 proteins
  131923271: 2 proteins
  130870042: 1 proteins
  129683862: 1 proteins
  128117329: 2 proteins
  127691891: 1 proteins
  127226408: 1 proteins
  127196445: 1 proteins
  126508837: 1 proteins
  125416387: 1 proteins
  125349910: 1 proteins
  124981321: 1 proteins
  124089763: 2 proteins
  122107421: 2 proteins
  121433520: 1 proteins
  119821037: 2 proteins
  118572924: 1 proteins
  117705848: 1 proteins
  116899475: 1 proteins
  116088543: 1 proteins
  114693636: 1 proteins
  114615022: 1 proteins
  114097823: 1 proteins
  113192262: 2 proteins
  110560533: 3 proteins
  110321694: 2 proteins
  110295410: 1 proteins
  109697806: 1 proteins
  107138667: 2 proteins
  105986543: 1 proteins
  104856565

* function: extract all protein ids per orthologs per genes into a list 

In [9]:
# create fasta folder
!mkdir fasta_rodentia

def flatten_protein_ids(variant_dict):
    """
    Convert:
        {"variant1": [ids], "variant2": [ids]}
    into one unique list of protein IDs for that gene.
    """
    seq_ids = []
    seen = set()

    for variant_id, protein_ids in variant_dict.items():
        for pid in protein_ids:
            if pid not in seen:
                seq_ids.append(pid)
                seen.add(pid)

    return seq_ids



* function: fetch protein details ('records') from NCBI
* modified

In [10]:
def fetch_protein_records(seq_ids):
    """
    Fetch GenBank protein records from NCBI.
    Uses chunks (at most 200) so the request does not become too large.
    """
    records = []
    chunk_size= 200
    for start in range(0, len(seq_ids), chunk_size):
        chunk = seq_ids[start:start + chunk_size]

        handle= Entrez.efetch(db="protein", rettype="gb", retmode="text", id=",".join(chunk))
        records.extend(list(SeqIO.parse(handle, "gb")))
        handle.close()

        time.sleep(0.35)

    return records

* function: get specie name from record eg 'Homo sapiens'

In [11]:
def get_species(record):
    """
    Retrieve species name from NCBI sequence record.

    Parameters
    ----------
    record : Bio.SeqRecord
        Sequence record.

    Returns
    -------
    str
        Species name.
    """
    description = record.description
    species = description.split(" [")[1][:-1]
    return species

* function; collect unique sequences; Homo sapiens first, then non Homo sapiens

In [12]:
def collect_unique_sequences_human_first(records):
    """
    Keep unique protein sequences.
    First collect unique Homo sapiens sequences,
    then collect unique non-human sequences.
    """
    seq_to_index = {}
    unique_records = []
    excluded_records = []

    inc = 0
    exc = 0

    # 1. Collect unique human sequences first
    for seq_record in records:
        sp = get_species(seq_record)
        if sp == "Homo sapiens":
            seq = str(seq_record.seq)

            if seq in seq_to_index:
                excluded_records.append(seq_record)
                print(f"\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                    f"\t*** same as sequence {seq_to_index[seq]} excluded ***")
                exc += 1
            else:
                seq_to_index[seq] = inc
                unique_records.append(seq_record)
                print(f"{inc}:\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}")
                inc += 1

    # 2. Collect other unique non-human sequences
    for seq_record in records:
        sp = get_species(seq_record)

        if sp != "Homo sapiens":
            seq = str(seq_record.seq)

            if seq in seq_to_index:
                excluded_records.append(seq_record)
                print(f"\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                    f"\t*** same as sequence {seq_to_index[seq]} excluded ***")
                exc += 1
            else:
                seq_to_index[seq] = inc
                unique_records.append(seq_record)
                print(
                    f"{inc}:\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                )
                inc += 1

    return unique_records, excluded_records, inc, exc


* function; make a dict ('sequence_dict') comprising protein and its sequence

In [13]:
def make_sequence_dict(variant_dict, records):
    """
    Preserve your original nested structure, but replace each protein ID
    with its actual protein sequence.
    """
    #seq_by_id = {seq_record.id: str(seq_record.seq) for seq_record in records}
    seq_by_id = {}
    for seq_record in records:
        seq_by_id[seq_record.id: str(seq_record.seq)]= {}

    sequence_dict = {}

    for variant_id, protein_ids in variant_dict.items():
        sequence_dict[variant_id] = {}

        for pid in protein_ids:
            sequence_dict[variant_id][pid] = seq_by_id.get(pid, None)

    return sequence_dict

* RUN CODE USING CREATED FUNCTIONS
* Store dictionary containing protein id, unique sequences, etc 'summary_df' as dataframe
* save sequences in fasta file per gene

In [15]:
protein_sequences = {}
unique_records_by_gene = {}
excluded_records_by_gene = {}

summary = []

for gene, variant_dict in result.items():

    print("\n" + "=" * 80) #demarcation line
    #print(datetime.now())
    print(f"{gene} orthologs")

    # Get all protein IDs for this gene from your result dictionary
    seq_ids = flatten_protein_ids(variant_dict)

    print(f"{gene}: {len(seq_ids)} protein IDs")
    print(f"Sequence IDs: {seq_ids[:10]}{' ...' if len(seq_ids) > 10 else ''}")

    if len(seq_ids) == 0:
        protein_sequences[gene] = {}
        unique_records_by_gene[gene] = []
        excluded_records_by_gene[gene] = []

        summary.append({
            "gene": gene,
            "n_protein_ids": 0,
            "n_records_fetched": 0,
            "n_unique_sequences": 0,
            "n_excluded_duplicates": 0,
            "fasta_file": None
        })

        continue

    # Fetch protein records from NCBI
    records = fetch_protein_records(seq_ids)

    print(f"{gene}: {len(records)} protein records fetched")

    # Store nested dictionary of actual sequences
    protein_sequences[gene] = make_sequence_dict(variant_dict, records)

    # Collect unique sequences, human first
    print(f"\n{gene} orthologs: unique primate sequences\n")

    unique_records, excluded_records, inc, exc = collect_unique_sequences_human_first(records)

    unique_records_by_gene[gene] = unique_records
    excluded_records_by_gene[gene] = excluded_records

    # Save FASTA file for that gene
    fasta_path = f"fasta/{gene}.fasta"

    with open(fasta_path, "w") as output:
        SeqIO.write(unique_records, output, "fasta")

    print(f"\nTotal: {inc} unique sequences, {exc} excluded")
    print(f"{fasta_path} saved!")

    summary.append({
        "gene": gene,
        "n_protein_ids": len(seq_ids),
        "n_records_fetched": len(records),
        "n_unique_sequences": inc,
        "n_excluded_duplicates": exc,
        "fasta_file": fasta_path
    })

summary_df = pd.DataFrame(summary)
summary_df


ABCD1 orthologs
ABCD1: 0 protein IDs
Sequence IDs: []

ALPL orthologs
ALPL: 0 protein IDs
Sequence IDs: []

AR orthologs
AR: 0 protein IDs
Sequence IDs: []

ATP7B orthologs
ATP7B: 0 protein IDs
Sequence IDs: []

BTK orthologs
BTK: 0 protein IDs
Sequence IDs: []

CASR orthologs
CASR: 0 protein IDs
Sequence IDs: []

CBS orthologs
CBS: 0 protein IDs
Sequence IDs: []

CFTR orthologs
CFTR: 0 protein IDs
Sequence IDs: []

CYBB orthologs
CYBB: 0 protein IDs
Sequence IDs: []

F7 orthologs
F7: 0 protein IDs
Sequence IDs: []

F8 orthologs
F8: 0 protein IDs
Sequence IDs: []

F9 orthologs
F9: 0 protein IDs
Sequence IDs: []

G6PD orthologs
G6PD: 0 protein IDs
Sequence IDs: []

GALT orthologs
GALT: 0 protein IDs
Sequence IDs: []

GBA1 orthologs
GBA1: 0 protein IDs
Sequence IDs: []

GJB1 orthologs
GJB1: 0 protein IDs
Sequence IDs: []

HBB orthologs
HBB: 0 protein IDs
Sequence IDs: []

HPRT1 orthologs
HPRT1: 0 protein IDs
Sequence IDs: []

IL2RG orthologs
IL2RG: 0 protein IDs
Sequence IDs: []

KCNH2 

,gene,n_protein_ids,n_records_fetched,n_unique_sequences,n_excluded_duplicates,fasta_file
0,ABCD1,0,0,0,0,None
1,ALPL,0,0,0,0,None
2,AR,0,0,0,0,None
3,ATP7B,0,0,0,0,None
4,BTK,0,0,0,0,None
5,CASR,0,0,0,0,None
6,CBS,0,0,0,0,None
7,CFTR,0,0,0,0,None
8,CYBB,0,0,0,0,None
9,F7,0,0,0,0,None
